In [6]:
import pandas as pd

path = "../data/raw/online_retail_II.xlsx"

sheets = pd.read_excel(path, sheet_name=None)   # loads all sheets as a dict
print(sheets.keys())

for name, df in sheets.items():
    print(name, df.shape)

dict_keys(['Year 2009-2010', 'Year 2010-2011'])
Year 2009-2010 (525461, 8)
Year 2010-2011 (541910, 8)


In [7]:
df = pd.concat(sheets.values(), ignore_index=True)
print(df.shape)
df.to_csv("../data/raw/online_retail_II.csv", index=False)

(1067371, 8)


In [8]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB


In [10]:
df.isna().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [11]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,1.067371e+06,1067371,1.067371e+06,824364.000000
mean,9.938898e+00,2011-01-02 21:13:55.394029,4.649388e+00,15324.638504
min,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000
25%,1.000000e+00,2010-07-09 09:46:00,1.250000e+00,13975.000000
50%,3.000000e+00,2010-12-07 15:28:00,2.100000e+00,15255.000000
75%,1.000000e+01,2011-07-22 10:23:00,4.150000e+00,16797.000000
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000
std,1.727058e+02,NaN,1.235531e+02,1697.464450


In [13]:
# These counts goes in Report
print((df["Quantity"] < 0).sum(), "rows with negative Quantity")
print((df["Price"] <= 0).sum(), "rows with Price <= 0")
print(df["Invoice"].astype(str).str.startswith("C").sum(), "cancelled invoices")

22950 rows with negative Quantity
6207 rows with Price <= 0
19494 cancelled invoices


In [ ]:
# Cleaning the data 
# Remove cancellations
clean = df.copy()
print(f"Start: {len(clean):,} rows")

clean = clean[~clean["Invoice"].astype(str).str.startswith("C")]
print(f"After removing cancellations: {len(clean):,} rows")

Start: 1,067,371 rows
After removing cancellations: 1,047,877 rows


In [15]:
# Remove wrong data of price and quantity
clean = clean[(clean["Quantity"] > 0) & (clean["Price"] > 0)]
print(f"After removing Quantity<=0 or Price<=0: {len(clean):,} rows")

After removing Quantity<=0 or Price<=0: 1,041,670 rows


In [17]:
# see nonProduct codes 
clean["StockCode"] = clean["StockCode"].astype(str).str.upper().str.strip()
is_product = clean["StockCode"].str.match(r"^\d{5}[A-Z]*$")
print(clean.loc[~is_product, "StockCode"].value_counts().head(20))

StockCode
POST            1886
DOT             1436
M                881
C2               272
ADJUST            36
BANK CHARGES      35
DCGS0058          30
GIFT_0001_20      26
GIFT_0001_30      24
DCGSSGIRL         23
DCGSSBOY          21
PADS              17
DCGS0076          15
GIFT_0001_10      14
DCGS0003          13
TEST001            9
GIFT_0001_50       6
DCGS0069           5
D                  5
GIFT_0001_40       5
Name: count, dtype: int64


In [18]:
# Remove non product Codes, and drop Customer ID
clean = clean[is_product]
print(f"After removing non-product codes: {len(clean):,} rows")

clean = clean.drop(columns=["Customer ID"])
print(clean.shape)
print(clean.isna().sum())

clean.to_csv("../data/processed/clean_transactions.csv", index=False)

After removing non-product codes: 1,036,877 rows
(1036877, 7)
Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Country        0
dtype: int64
